# Entrenamiento YOLO en Colab (datasets separados)

Este notebook entrena **YOLO11n**, **YOLO11s** y **YOLO26** en secciones separadas.

- `YOLO11n` y `YOLO11s` usan el dataset YOLO11
- `YOLO26` usa el dataset YOLO26
- Cada dataset se divide por separado en `train / val / test`
- Aplica amplificacion con **Albumentations** (opcional)
- Entrena un modelo por vez con `patience` configurable


## 1) Instalacion

In [ ]:
!pip install -q ultralytics albumentations opencv-python pyyaml
import os, shutil, random, yaml, glob
from pathlib import Path
import cv2
import albumentations as A
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')
print('Ultralytics listo')

## 2) Configuracion de rutas, split y entrenamiento

In [ ]:
# Ajusta solo si cambia la ubicacion en tu Drive
# Dataset para YOLO11n y YOLO11s
DATASET_YOLO11 = '/content/drive/MyDrive/Cards'
# Dataset para YOLO26
DATASET_YOLO26 = '/content/drive/MyDrive/Cards-2'

# Carpeta de trabajo en Colab
BASE_DIR = Path('/content/cards_yolo')
YOLO11_SPLIT_DIR = BASE_DIR / 'yolo11_split_dataset'
YOLO26_SPLIT_DIR = BASE_DIR / 'yolo26_split_dataset'
YOLO11_AUG_DIR = BASE_DIR / 'yolo11_split_dataset_aug'
YOLO26_AUG_DIR = BASE_DIR / 'yolo26_split_dataset_aug'
RUNS_DIR = BASE_DIR / 'runs'

# Split
VAL_RATIO = 0.15
TEST_RATIO = 0.10
SEED = 42

# Entrenamiento
IMG_SIZE = 640
EPOCHS = 120
BATCH = 16
PATIENCE = 25   # <- limite de paciencia
WORKERS = 2

# Cambia a True si quieres aumentar dataset offline con Albumentations
USE_OFFLINE_ALBUMENTATIONS = True
AUG_MULTIPLIER = 1  # copias sinteticas por imagen de train

random.seed(SEED)
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('Configuracion aplicada')

## 3) Utilidades

In [ ]:
def read_yolo_labels(label_path):
    bboxes, class_ids = [], []
    if not Path(label_path).exists():
        return bboxes, class_ids
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, x, y, w, h = parts
            class_ids.append(int(cls))
            bboxes.append([float(x), float(y), float(w), float(h)])
    return bboxes, class_ids

def write_yolo_labels(label_path, bboxes, class_ids):
    with open(label_path, 'w') as f:
        for cls, box in zip(class_ids, bboxes):
            x, y, w, h = box
            f.write(f'{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')

def ensure_split_structure(root):
    for s in ['train', 'val', 'test']:
        (root / s / 'images').mkdir(parents=True, exist_ok=True)
        (root / s / 'labels').mkdir(parents=True, exist_ok=True)

def copy_pair(img_src, lbl_src, out_img, out_lbl):
    shutil.copy2(img_src, out_img)
    shutil.copy2(lbl_src, out_lbl)

def load_names_from_yaml(dataset_yaml):
    with open(dataset_yaml, 'r') as f:
        data = yaml.safe_load(f)
    return data['names']

def split_single_dataset(dataset_path, output_dir, val_ratio=0.15, test_ratio=0.10, seed=42):
    ensure_split_structure(output_dir)
    pairs = []
    images = sorted(glob.glob(f'{dataset_path}/train/images/*'))
    for img in images:
        stem = Path(img).stem
        lbl = f'{dataset_path}/train/labels/{stem}.txt'
        if Path(lbl).exists():
            pairs.append((img, lbl))

    rnd = random.Random(seed)
    rnd.shuffle(pairs)

    n_total = len(pairs)
    n_test = int(n_total * test_ratio)
    n_val = int(n_total * val_ratio)
    n_train = n_total - n_val - n_test

    splits = {
        'train': pairs[:n_train],
        'val': pairs[n_train:n_train + n_val],
        'test': pairs[n_train + n_val:]
    }

    ds_name = Path(dataset_path).name
    for split_name, items in splits.items():
        for i, (img, lbl) in enumerate(items):
            ext = Path(img).suffix.lower()
            out_stem = f'{ds_name}_{Path(img).stem}_{i}'
            out_img = output_dir / split_name / 'images' / f'{out_stem}{ext}'
            out_lbl = output_dir / split_name / 'labels' / f'{out_stem}.txt'
            copy_pair(img, lbl, out_img, out_lbl)

    return {
        'train': len(splits['train']),
        'val': len(splits['val']),
        'test': len(splits['test'])
    }

def apply_offline_augment(split_dir, aug_dir, aug_multiplier=1):
    if aug_dir.exists():
        shutil.rmtree(aug_dir)
    shutil.copytree(split_dir, aug_dir)

    transform = A.Compose([
        A.RandomBrightnessContrast(p=0.35),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.30),
        A.GaussNoise(p=0.20),
        A.MotionBlur(blur_limit=3, p=0.15),
        A.Affine(scale=(0.90, 1.10), translate_percent=(0.0, 0.04), rotate=(-8, 8), shear=(-3, 3), p=0.40),
        A.RandomShadow(p=0.15),
        A.CLAHE(p=0.15)
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

    train_img_dir = aug_dir / 'train' / 'images'
    train_lbl_dir = aug_dir / 'train' / 'labels'
    imgs = sorted(train_img_dir.glob('*'))
    created = 0

    for img_path in imgs:
        lbl_path = train_lbl_dir / f'{img_path.stem}.txt'
        bboxes, cls_ids = read_yolo_labels(lbl_path)
        if len(bboxes) == 0:
            continue

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        for k in range(aug_multiplier):
            aug = transform(image=image, bboxes=bboxes, class_labels=cls_ids)
            out_img_name = f'{img_path.stem}_aug{k}.jpg'
            out_lbl_name = f'{img_path.stem}_aug{k}.txt'
            out_img_path = train_img_dir / out_img_name
            out_lbl_path = train_lbl_dir / out_lbl_name

            out = cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(out_img_path), out)
            write_yolo_labels(out_lbl_path, aug['bboxes'], aug['class_labels'])
            created += 1

    return created

def build_yaml(data_root, names, yaml_path):
    final_yaml = {
        'path': str(data_root),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(names),
        'names': names
    }
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(final_yaml, f, sort_keys=False)


## 4) Split por separado (YOLO11 y YOLO26)

In [ ]:
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

names_yolo11 = load_names_from_yaml(f'{DATASET_YOLO11}/data.yaml')
names_yolo26 = load_names_from_yaml(f'{DATASET_YOLO26}/data.yaml')

print(f'Numero de clases YOLO11 dataset: {len(names_yolo11)}')
print(f'Numero de clases YOLO26 dataset: {len(names_yolo26)}')

stats_11 = split_single_dataset(
    dataset_path=DATASET_YOLO11,
    output_dir=YOLO11_SPLIT_DIR,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED
)

stats_26 = split_single_dataset(
    dataset_path=DATASET_YOLO26,
    output_dir=YOLO26_SPLIT_DIR,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED
)

print('Split YOLO11 ->', stats_11)
print('Split YOLO26 ->', stats_26)

## 5) Albumentations offline por dataset (opcional)

Se aplica solo a `train` para no contaminar `val/test`, y se procesa por separado para YOLO11 y YOLO26.

In [ ]:
if USE_OFFLINE_ALBUMENTATIONS:
    created_11 = apply_offline_augment(YOLO11_SPLIT_DIR, YOLO11_AUG_DIR, AUG_MULTIPLIER)
    created_26 = apply_offline_augment(YOLO26_SPLIT_DIR, YOLO26_AUG_DIR, AUG_MULTIPLIER)
    DATA_ROOT_YOLO11 = YOLO11_AUG_DIR
    DATA_ROOT_YOLO26 = YOLO26_AUG_DIR
    print(f'Albumentations YOLO11 activo. Imagenes nuevas creadas: {created_11}')
    print(f'Albumentations YOLO26 activo. Imagenes nuevas creadas: {created_26}')
else:
    DATA_ROOT_YOLO11 = YOLO11_SPLIT_DIR
    DATA_ROOT_YOLO26 = YOLO26_SPLIT_DIR
    print('Albumentations offline desactivado.')

print('YOLO11 train:', len(list((DATA_ROOT_YOLO11 / 'train' / 'images').glob('*'))))
print('YOLO11 val  :', len(list((DATA_ROOT_YOLO11 / 'val' / 'images').glob('*'))))
print('YOLO11 test :', len(list((DATA_ROOT_YOLO11 / 'test' / 'images').glob('*'))))

print('YOLO26 train:', len(list((DATA_ROOT_YOLO26 / 'train' / 'images').glob('*'))))
print('YOLO26 val  :', len(list((DATA_ROOT_YOLO26 / 'val' / 'images').glob('*'))))
print('YOLO26 test :', len(list((DATA_ROOT_YOLO26 / 'test' / 'images').glob('*'))))

## 6) Crear data.yaml por modelo

In [ ]:
YOLO11_DATA_YAML = BASE_DIR / 'cards_yolo11.yaml'
YOLO26_DATA_YAML = BASE_DIR / 'cards_yolo26.yaml'

build_yaml(DATA_ROOT_YOLO11, names_yolo11, YOLO11_DATA_YAML)
build_yaml(DATA_ROOT_YOLO26, names_yolo26, YOLO26_DATA_YAML)

print('YAML YOLO11:', YOLO11_DATA_YAML)
print(open(YOLO11_DATA_YAML).read())
print('YAML YOLO26:', YOLO26_DATA_YAML)
print(open(YOLO26_DATA_YAML).read())

## 7) Seccion A - Entrenar YOLO11n

In [ ]:
model_11n = YOLO('yolo11n.pt')
results_11n = model_11n.train(
    data=str(YOLO11_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo11n_cards',
    plots=True,
    device=0,
    cache=True,
    amp=True,
    close_mosaic=10
)

## 8) Seccion B - Entrenar YOLO11s

In [ ]:
model_11s = YOLO('yolo11s.pt')
results_11s = model_11s.train(
    data=str(YOLO11_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo11s_cards',
    plots=True,
    device=0,
    cache=True,
    amp=True,
    close_mosaic=10
)

## 9) Seccion C - Entrenar YOLO26

Si tu instalacion de Ultralytics trae pesos `yolo26.pt`, este bloque corre directo.
Si no, cambia el nombre del peso base por el disponible en tu entorno.

In [ ]:
BASE_WEIGHT_YOLO26 = 'yolo26.pt'  # cambia si usas otro nombre de peso
model_26 = YOLO(BASE_WEIGHT_YOLO26)
results_26 = model_26.train(
    data=str(YOLO26_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo26_cards',
    plots=True,
    device=0,
    cache=True,
    amp=True,
    close_mosaic=10
)

## 10) Evaluar y exportar mejor modelo

In [ ]:
# Ejemplo con el ultimo modelo entrenado (YOLO26)
metrics = model_26.val(data=str(YOLO26_DATA_YAML), split='test')
print(metrics)

# Ejemplos alternos:
# metrics_11n = model_11n.val(data=str(YOLO11_DATA_YAML), split='test')
# metrics_11s = model_11s.val(data=str(YOLO11_DATA_YAML), split='test')

# Export opcional
# model_26.export(format='onnx')

## Recomendaciones rapidas

- Si hay overfitting: sube `AUG_MULTIPLIER` a `2`, deja `PATIENCE` entre `20` y `30`.
- Si mAP no sube: prueba `IMG_SIZE=768` con batch menor.
- Compara modelos por `mAP50-95` y latencia de inferencia.
- Mantener `val/test` sin augment synthetic.
- Entrena cada seccion por separado para no mezclar corridas.
